# Support Vector Machines from Scratch

**Goal:** Implement SVMs from first principles -- hard-margin, soft-margin, kernel trick, and simplified SMO -- with strong geometric intuition.

---

## Core Intuition

SVM finds the **maximum margin** hyperplane separating two classes. The margin is the distance between the decision boundary and the nearest data points from each class.

- **Decision boundary:** $\mathbf{w} \cdot \mathbf{x} + b = 0$
- **Margin:** $\frac{2}{\|\mathbf{w}\|}$ (distance between $\mathbf{w} \cdot \mathbf{x} + b = +1$ and $\mathbf{w} \cdot \mathbf{x} + b = -1$)
- **Support vectors:** Points on the margin boundary. Only these determine the decision boundary.
- **Maximize margin** = minimize $\|\mathbf{w}\|^2$ subject to $y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1$

### Hinge Loss Formulation
$$\min_{\mathbf{w}, b} \frac{\lambda}{2}\|\mathbf{w}\|^2 + \frac{1}{n}\sum_{i=1}^{n} \max(0, 1 - y_i(\mathbf{w} \cdot \mathbf{x}_i + b))$$

This is equivalent to the constrained optimization via Lagrangian duality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Generate Linearly Separable Data

In [ ]:
def make_linear_data(n=100, sep=1.5, seed=42):
    """Generate linearly separable 2D data."""
    rng = np.random.RandomState(seed)
    X_pos = rng.randn(n // 2, 2) + np.array([sep, sep])
    X_neg = rng.randn(n // 2, 2) + np.array([-sep, -sep])
    X = np.vstack([X_pos, X_neg])
    y = np.array([1] * (n // 2) + [-1] * (n // 2))
    return X, y

X_lin, y_lin = make_linear_data(200, sep=1.5)

plt.figure(figsize=(8, 6))
plt.scatter(X_lin[y_lin == 1, 0], X_lin[y_lin == 1, 1], c='blue', marker='o', label='+1', alpha=0.6)
plt.scatter(X_lin[y_lin == -1, 0], X_lin[y_lin == -1, 1], c='red', marker='x', label='-1', alpha=0.6)
plt.legend()
plt.title('Linearly Separable Data')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.show()

## 2. Hard-Margin SVM via Gradient Descent on Hinge Loss

For linearly separable data, we can use hinge loss with strong regularization (large $\lambda$):

**Hinge loss gradient:**
- If $y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1$: $\nabla_w = \lambda \mathbf{w}$, $\nabla_b = 0$
- If $y_i(\mathbf{w} \cdot \mathbf{x}_i + b) < 1$: $\nabla_w = \lambda \mathbf{w} - y_i \mathbf{x}_i$, $\nabla_b = -y_i$

In [ ]:
class HardMarginSVM:
    """Hard-margin SVM using sub-gradient descent on hinge loss.
    
    Works only on linearly separable data.
    Uses high regularization to approximate hard margin.
    """
    
    def __init__(self, lr=0.001, lambda_param=0.01, n_iters=1000):
        self.lr = lr
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None
        self.b = None
        self.losses = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0
        self.losses = []
        
        for epoch in range(self.n_iters):
            margins = y * (X @ self.w + self.b)
            
            # Compute loss
            hinge = np.maximum(0, 1 - margins)
            loss = self.lambda_param / 2 * np.dot(self.w, self.w) + np.mean(hinge)
            self.losses.append(loss)
            
            # Sub-gradient descent
            dw = self.lambda_param * self.w.copy()
            db = 0.0
            
            violated = margins < 1  # hinge is active
            if violated.any():
                dw -= np.mean(y[violated, None] * X[violated], axis=0)
                db -= np.mean(y[violated])
            
            self.w -= self.lr * dw
            self.b -= self.lr * db
        
        return self
    
    def decision_function(self, X):
        return X @ self.w + self.b
    
    def predict(self, X):
        return np.sign(self.decision_function(X))
    
    def margin(self):
        return 2.0 / np.linalg.norm(self.w)

# Train
hard_svm = HardMarginSVM(lr=0.001, lambda_param=0.01, n_iters=2000)
hard_svm.fit(X_lin, y_lin)

acc = np.mean(hard_svm.predict(X_lin) == y_lin)
print(f"Accuracy: {acc:.4f}")
print(f"Margin: {hard_svm.margin():.4f}")
print(f"w = {hard_svm.w}, b = {hard_svm.b:.4f}")

In [ ]:
def plot_svm_boundary(X, y, svm, title='SVM Decision Boundary', ax=None):
    """Plot decision boundary, margins, and support vectors for linear SVM."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    # Plot data
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='blue', marker='o', label='+1', alpha=0.6)
    ax.scatter(X[y == -1, 0], X[y == -1, 1], c='red', marker='x', label='-1', alpha=0.6)
    
    # Decision boundary and margins
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    
    xx = np.linspace(xlim[0], xlim[1], 200)
    yy = np.linspace(ylim[0], ylim[1], 200)
    XX, YY = np.meshgrid(xx, yy)
    Z = svm.decision_function(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
    
    ax.contour(XX, YY, Z, levels=[-1, 0, 1], colors=['red', 'black', 'blue'],
               linestyles=['--', '-', '--'], linewidths=[1.5, 2, 1.5])
    ax.contourf(XX, YY, Z, levels=[-1, 1], colors=['yellow'], alpha=0.15)
    
    # Highlight support vectors (points near the margin)
    margins = y * svm.decision_function(X)
    sv_mask = margins <= 1.05  # within or on margin
    ax.scatter(X[sv_mask, 0], X[sv_mask, 1], s=100, facecolors='none',
               edgecolors='green', linewidths=2, label='Support vectors')
    
    ax.set_title(f'{title}\nMargin = {svm.margin():.3f}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    return ax

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_svm_boundary(X_lin, y_lin, hard_svm, 'Hard-Margin SVM', axes[0])
axes[1].plot(hard_svm.losses, 'b-', linewidth=1)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Loss')
axes[1].set_title('Training Loss')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Soft-Margin SVM (C Parameter)

Real data is rarely linearly separable. Soft-margin SVM allows some violations:

$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{n} \max(0, 1 - y_i(\mathbf{w} \cdot \mathbf{x}_i + b))$$

- **Large C:** Penalize violations heavily -> narrow margin, fewer violations (risk of overfitting)
- **Small C:** Allow more violations -> wider margin, more robust (risk of underfitting)
- Equivalent to $\lambda = \frac{1}{2nC}$ in the regularized hinge loss

In [ ]:
class SoftMarginSVM:
    """Soft-margin SVM with C parameter, trained via SGD."""
    
    def __init__(self, C=1.0, lr=0.001, n_iters=1000):
        self.C = C
        self.lr = lr
        self.n_iters = n_iters
        self.w = None
        self.b = None
        self.losses = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0
        self.losses = []
        
        for epoch in range(self.n_iters):
            margins = y * (X @ self.w + self.b)
            hinge = np.maximum(0, 1 - margins)
            loss = 0.5 * np.dot(self.w, self.w) + self.C * np.mean(hinge)
            self.losses.append(loss)
            
            # Gradient
            dw = self.w.copy()
            db = 0.0
            
            violated = margins < 1
            if violated.any():
                dw -= self.C * np.mean(y[violated, None] * X[violated], axis=0)
                db -= self.C * np.mean(y[violated])
            
            self.w -= self.lr * dw
            self.b -= self.lr * db
        
        return self
    
    def decision_function(self, X):
        return X @ self.w + self.b
    
    def predict(self, X):
        return np.sign(self.decision_function(X))
    
    def margin(self):
        return 2.0 / (np.linalg.norm(self.w) + 1e-10)

# Make data with some overlap
X_soft, y_soft = make_linear_data(200, sep=1.0, seed=42)

# Compare different C values
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, C_val in zip(axes, [0.01, 1.0, 100.0]):
    svm = SoftMarginSVM(C=C_val, lr=0.001, n_iters=2000)
    svm.fit(X_soft, y_soft)
    acc = np.mean(svm.predict(X_soft) == y_soft)
    plot_svm_boundary(X_soft, y_soft, svm, f'C={C_val} (acc={acc:.2f})', ax)

plt.suptitle('Effect of C Parameter: Underfitting <-> Overfitting', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Kernel Trick

The kernel trick maps data to a higher-dimensional space **implicitly** via a kernel function:
$$K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i) \cdot \phi(\mathbf{x}_j)$$

We never compute $\phi(\mathbf{x})$ explicitly! We only need dot products.

Common kernels:
- **Linear:** $K(x, y) = x \cdot y$
- **Polynomial:** $K(x, y) = (x \cdot y + c)^d$
- **RBF (Gaussian):** $K(x, y) = \exp(-\gamma \|x - y\|^2)$  (maps to infinite-dimensional space!)

In [ ]:
def linear_kernel(X1, X2):
    return X1 @ X2.T

def polynomial_kernel(X1, X2, degree=3, coef0=1.0):
    return (X1 @ X2.T + coef0) ** degree

def rbf_kernel(X1, X2, gamma=1.0):
    """RBF kernel: exp(-gamma * ||x1 - x2||^2)"""
    sq1 = np.sum(X1 ** 2, axis=1).reshape(-1, 1)
    sq2 = np.sum(X2 ** 2, axis=1).reshape(1, -1)
    dist_sq = sq1 + sq2 - 2 * X1 @ X2.T
    return np.exp(-gamma * dist_sq)

# Visualize what RBF kernel does
from sklearn.datasets import make_circles
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, factor=0.3, random_state=42)
y_circles = 2 * y_circles - 1  # convert to {-1, +1}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Original space
axes[0].scatter(X_circles[y_circles == 1, 0], X_circles[y_circles == 1, 1], c='blue', alpha=0.6)
axes[0].scatter(X_circles[y_circles == -1, 0], X_circles[y_circles == -1, 1], c='red', alpha=0.6)
axes[0].set_title('Original Space (not linearly separable)')

# Feature mapping: add ||x||^2 as third dimension
X_lifted = np.c_[X_circles, np.sum(X_circles**2, axis=1)]
ax3d = fig.add_subplot(132, projection='3d')
ax3d.scatter(X_lifted[y_circles == 1, 0], X_lifted[y_circles == 1, 1], X_lifted[y_circles == 1, 2], c='blue', alpha=0.6)
ax3d.scatter(X_lifted[y_circles == -1, 0], X_lifted[y_circles == -1, 1], X_lifted[y_circles == -1, 2], c='red', alpha=0.6)
ax3d.set_title('Lifted to 3D (separable!)')
axes[1].set_visible(False)

# Kernel matrix heatmap
K = rbf_kernel(X_circles, X_circles, gamma=5.0)
# Sort by label for visualization
sort_idx = np.argsort(y_circles)
axes[2].imshow(K[np.ix_(sort_idx, sort_idx)], cmap='viridis')
axes[2].set_title('RBF Kernel Matrix (sorted by class)')
axes[2].set_xlabel('Samples')
axes[2].set_ylabel('Samples')

plt.tight_layout()
plt.show()

## 5. Simplified SMO (Sequential Minimal Optimization)

SMO solves the dual SVM problem by optimizing two Lagrange multipliers at a time. This is the standard SVM solver.

**Dual problem:**
$$\max_{\alpha} \sum_i \alpha_i - \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j y_i y_j K(x_i, x_j)$$
subject to $0 \leq \alpha_i \leq C$ and $\sum_i \alpha_i y_i = 0$

Support vectors have $\alpha_i > 0$.

In [ ]:
class KernelSVM:
    """Kernel SVM using simplified SMO algorithm."""
    
    def __init__(self, C=1.0, kernel='rbf', gamma=1.0, degree=3, coef0=1.0, tol=1e-3, max_passes=100):
        self.C = C
        self.kernel_name = kernel
        self.gamma = gamma
        self.degree = degree
        self.coef0 = coef0
        self.tol = tol
        self.max_passes = max_passes
    
    def _kernel(self, X1, X2):
        if self.kernel_name == 'linear':
            return linear_kernel(X1, X2)
        elif self.kernel_name == 'poly':
            return polynomial_kernel(X1, X2, self.degree, self.coef0)
        elif self.kernel_name == 'rbf':
            return rbf_kernel(X1, X2, self.gamma)
        else:
            raise ValueError(f"Unknown kernel: {self.kernel_name}")
    
    def fit(self, X, y):
        n = X.shape[0]
        self.X_train = X.copy()
        self.y_train = y.copy().astype(float)
        
        # Precompute kernel matrix
        K = self._kernel(X, X)
        
        # Initialize alphas and bias
        self.alphas = np.zeros(n)
        self.b = 0.0
        
        passes = 0
        while passes < self.max_passes:
            num_changed = 0
            
            for i in range(n):
                # Compute error for i
                Ei = np.sum(self.alphas * self.y_train * K[i]) + self.b - self.y_train[i]
                
                # Check KKT violations
                if ((self.y_train[i] * Ei < -self.tol and self.alphas[i] < self.C) or
                    (self.y_train[i] * Ei > self.tol and self.alphas[i] > 0)):
                    
                    # Pick j randomly, j != i
                    j = i
                    while j == i:
                        j = np.random.randint(n)
                    
                    Ej = np.sum(self.alphas * self.y_train * K[j]) + self.b - self.y_train[j]
                    
                    alpha_i_old = self.alphas[i]
                    alpha_j_old = self.alphas[j]
                    
                    # Compute L and H bounds
                    if self.y_train[i] != self.y_train[j]:
                        L = max(0, self.alphas[j] - self.alphas[i])
                        H = min(self.C, self.C + self.alphas[j] - self.alphas[i])
                    else:
                        L = max(0, self.alphas[i] + self.alphas[j] - self.C)
                        H = min(self.C, self.alphas[i] + self.alphas[j])
                    
                    if L >= H:
                        continue
                    
                    # Compute eta
                    eta = 2 * K[i, j] - K[i, i] - K[j, j]
                    if eta >= 0:
                        continue
                    
                    # Update alpha_j
                    self.alphas[j] -= self.y_train[j] * (Ei - Ej) / eta
                    self.alphas[j] = np.clip(self.alphas[j], L, H)
                    
                    if abs(self.alphas[j] - alpha_j_old) < 1e-5:
                        continue
                    
                    # Update alpha_i
                    self.alphas[i] += self.y_train[i] * self.y_train[j] * (alpha_j_old - self.alphas[j])
                    
                    # Update bias
                    b1 = (self.b - Ei - self.y_train[i] * (self.alphas[i] - alpha_i_old) * K[i, i]
                           - self.y_train[j] * (self.alphas[j] - alpha_j_old) * K[i, j])
                    b2 = (self.b - Ej - self.y_train[i] * (self.alphas[i] - alpha_i_old) * K[i, j]
                           - self.y_train[j] * (self.alphas[j] - alpha_j_old) * K[j, j])
                    
                    if 0 < self.alphas[i] < self.C:
                        self.b = b1
                    elif 0 < self.alphas[j] < self.C:
                        self.b = b2
                    else:
                        self.b = (b1 + b2) / 2
                    
                    num_changed += 1
            
            if num_changed == 0:
                passes += 1
            else:
                passes = 0
        
        # Store support vectors
        self.support_mask = self.alphas > 1e-5
        self.n_support = np.sum(self.support_mask)
        return self
    
    def decision_function(self, X):
        K = self._kernel(X, self.X_train)
        return K @ (self.alphas * self.y_train) + self.b
    
    def predict(self, X):
        return np.sign(self.decision_function(X))

# Test on circles data
svm_rbf = KernelSVM(C=10.0, kernel='rbf', gamma=5.0, max_passes=50)
svm_rbf.fit(X_circles, y_circles)
acc = np.mean(svm_rbf.predict(X_circles) == y_circles)
print(f"RBF SVM accuracy: {acc:.4f}")
print(f"Number of support vectors: {svm_rbf.n_support}")

In [ ]:
def plot_kernel_svm(X, y, svm, title, ax):
    """Plot decision boundary for kernel SVM."""
    h = 0.05
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    Z = svm.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, levels=50, cmap='RdYlBu', alpha=0.6)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors='k', linestyles=['--', '-', '--'])
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='blue', marker='o', edgecolors='k', s=30)
    ax.scatter(X[y == -1, 0], X[y == -1, 1], c='red', marker='x', s=30)
    
    # Highlight support vectors
    if hasattr(svm, 'support_mask'):
        sv = X[svm.support_mask]
        ax.scatter(sv[:, 0], sv[:, 1], s=100, facecolors='none',
                   edgecolors='lime', linewidths=2)
    
    ax.set_title(title)

# Compare kernels on circles data
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

kernels = [
    ('linear', {'C': 1.0, 'kernel': 'linear'}),
    ('poly (d=3)', {'C': 1.0, 'kernel': 'poly', 'degree': 3}),
    ('rbf (gamma=5)', {'C': 10.0, 'kernel': 'rbf', 'gamma': 5.0}),
]

for ax, (name, params) in zip(axes, kernels):
    svm_temp = KernelSVM(**params, max_passes=50)
    svm_temp.fit(X_circles, y_circles)
    acc = np.mean(svm_temp.predict(X_circles) == y_circles)
    plot_kernel_svm(X_circles, y_circles, svm_temp,
                    f'{name}\nacc={acc:.2f}, SVs={svm_temp.n_support}', ax)

plt.suptitle('Kernel SVM on Non-Linear Data (circles)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Effect of C Parameter Visualization

In [ ]:
from sklearn.datasets import make_moons
X_moons, y_moons = make_moons(n_samples=200, noise=0.2, random_state=42)
y_moons = 2 * y_moons - 1

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
C_values = [0.01, 0.1, 1.0, 100.0]

for ax, C_val in zip(axes, C_values):
    svm_temp = KernelSVM(C=C_val, kernel='rbf', gamma=2.0, max_passes=30)
    svm_temp.fit(X_moons, y_moons)
    acc = np.mean(svm_temp.predict(X_moons) == y_moons)
    plot_kernel_svm(X_moons, y_moons, svm_temp,
                    f'C={C_val}, acc={acc:.2f}', ax)

plt.suptitle('Effect of C: Small C = smooth boundary (underfit), Large C = complex boundary (overfit)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Compare with sklearn SVM

In [ ]:
from sklearn.svm import SVC

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Our SVM
our_svm = KernelSVM(C=10.0, kernel='rbf', gamma=2.0, max_passes=50)
our_svm.fit(X_moons, y_moons)
acc_ours = np.mean(our_svm.predict(X_moons) == y_moons)
plot_kernel_svm(X_moons, y_moons, our_svm,
                f'Our SVM (acc={acc_ours:.3f}, SVs={our_svm.n_support})', axes[0])

# sklearn SVM
sk_svm = SVC(C=10.0, kernel='rbf', gamma=2.0)
sk_svm.fit(X_moons, y_moons)
acc_sk = sk_svm.score(X_moons, y_moons)

h = 0.05
x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = sk_svm.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[1].contourf(xx, yy, Z, levels=50, cmap='RdYlBu', alpha=0.6)
axes[1].contour(xx, yy, Z, levels=[-1, 0, 1], colors='k', linestyles=['--', '-', '--'])
axes[1].scatter(X_moons[y_moons == 1, 0], X_moons[y_moons == 1, 1], c='blue', marker='o', edgecolors='k', s=30)
axes[1].scatter(X_moons[y_moons == -1, 0], X_moons[y_moons == -1, 1], c='red', marker='x', s=30)
sv = X_moons[sk_svm.support_]
axes[1].scatter(sv[:, 0], sv[:, 1], s=100, facecolors='none', edgecolors='lime', linewidths=2)
axes[1].set_title(f'sklearn SVM (acc={acc_sk:.3f}, SVs={len(sk_svm.support_)})')

plt.suptitle('Our SMO vs sklearn SVC', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Interview Questions

### "What is the dual formulation and why?"
The dual converts the constrained primal problem into an unconstrained problem over Lagrange multipliers $\alpha_i$. Benefits:
1. The dual depends on data only through dot products $x_i \cdot x_j$, enabling the **kernel trick**.
2. The number of variables equals the number of training samples (vs. number of features in primal).
3. The KKT conditions give complementary slackness: $\alpha_i > 0$ only for support vectors.

### "Why do only support vectors matter?"
By KKT complementarity, $\alpha_i(y_i(w \cdot x_i + b) - 1 + \xi_i) = 0$. So either $\alpha_i = 0$ (point doesn't influence the solution) or the constraint is active (point is on/inside the margin = support vector). The decision boundary is entirely determined by support vectors.

### "SVM vs Logistic Regression?"
- Both are linear classifiers (without kernels).
- **Loss:** SVM uses hinge loss (flat at 0 beyond margin), LR uses log loss (always positive).
- **Sparsity:** SVM has sparse support (only SVs matter); LR uses all points.
- **Probabilistic:** LR gives calibrated probabilities; SVM gives distances, not probabilities.
- **Kernels:** SVM naturally extends to kernels via the dual; LR can use kernels but less common.
- In practice, performance is often similar for linear models.

### "How does the kernel trick avoid computing in high dimensions?"
The kernel function $K(x_i, x_j) = \phi(x_i) \cdot \phi(x_j)$ computes the dot product in the high-dimensional feature space without ever explicitly computing $\phi(x)$. For RBF, the feature space is infinite-dimensional, but the kernel is just $\exp(-\gamma\|x-y\|^2)$, which is O(d) to compute.

### Notes: Hinge Loss vs Cross-Entropy
- Hinge: $\max(0, 1 - yf(x))$ -- zero loss once correctly classified with margin > 1
- Cross-entropy: $\log(1 + e^{-yf(x)})$ -- always positive, never zero
- Hinge creates sparser solutions; cross-entropy gives smoother probability estimates
- Both are convex upper bounds on 0-1 loss

In [ ]:
# Visualize hinge loss vs cross-entropy vs 0-1 loss
z = np.linspace(-3, 3, 500)
hinge = np.maximum(0, 1 - z)
logistic = np.log(1 + np.exp(-z))
zero_one = (z < 0).astype(float)

plt.figure(figsize=(8, 5))
plt.plot(z, hinge, 'b-', linewidth=2, label='Hinge loss')
plt.plot(z, logistic, 'r-', linewidth=2, label='Log loss')
plt.plot(z, zero_one, 'k--', linewidth=2, label='0-1 loss')
plt.xlabel('y * f(x) (margin)')
plt.ylabel('Loss')
plt.title('Loss Functions for Classification')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(-0.1, 3)
plt.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
plt.axvline(x=1, color='gray', linestyle=':', alpha=0.3, label='margin=1')
plt.tight_layout()
plt.show()

In [ ]:
print("Notebook 09 complete: SVM from Scratch")
print("="*50)
print("Implemented:")
print("  - Hard-margin SVM (sub-gradient descent)")
print("  - Soft-margin SVM (C parameter)")
print("  - Kernel functions: linear, polynomial, RBF")
print("  - Simplified SMO algorithm (dual)")
print("  - Validated against sklearn SVC")